# Predator-Prey MARL Exploration

This notebook lets you interactively explore the environment, agents, and training results.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from predator_prey.env import PredatorPreyEnv
from predator_prey.agents import RandomAgent, IQLAgent, VDNAgent

## 1. Environment Basics

Let's create the environment and inspect observations, actions, and a single episode.

In [ ]:
env = PredatorPreyEnv(grid_size=7, max_steps=100)
obs = env.reset(seed=42)

print("Agent names:", env.agents)
print("Number of actions:", env.n_actions)
print("\nInitial observations:")
for name, o in obs.items():
    print(f"  {name}: {o}  (self_r, self_c, other_dr, other_dc, prey_dr, prey_dc)")

print("\nInitial grid:")
print(env.render())

In [ ]:
# Run a single episode with random actions
env = PredatorPreyEnv()
obs = env.reset(seed=0)
done = False
step = 0

while not done:
    actions = {name: np.random.randint(5) for name in env.agents}
    obs, rewards, dones, infos = env.step(actions)
    done = dones["__all__"]
    step += 1

print(f"Episode finished in {step} steps")
print(f"Captured: {infos['captured']}")
print(f"Cooperative capture: {infos['cooperative_capture']}")

## 2. Grid Visualization (matplotlib)

Simple matplotlib visualization for the notebook (no pygame needed).

In [ ]:
def plot_grid(grid, ax=None, title=""):
    """Plot the grid state with matplotlib."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    size = grid.shape[0]
    ax.set_xlim(-0.5, size - 0.5)
    ax.set_ylim(-0.5, size - 0.5)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.grid(True, linewidth=0.5)
    ax.set_title(title)

    for r in range(size):
        for c in range(size):
            if grid[r, c] == 1:
                ax.plot(c, r, 'ro', markersize=20)
            elif grid[r, c] == 2:
                ax.plot(c, r, 'bs', markersize=20)
            elif grid[r, c] == 3:
                ax.plot(c, r, 'mp', markersize=25)
    return ax

# Show initial state of 4 random episodes
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
env = PredatorPreyEnv()
for i, ax in enumerate(axes):
    env.reset(seed=i)
    plot_grid(env.render(), ax=ax, title=f"Seed {i}")
plt.suptitle("Red=Predator, Blue=Prey", y=1.02)
plt.tight_layout()

## 3. Random Baseline Statistics

Run many episodes with random agents and look at the capture rate distribution.

In [ ]:
from predator_prey.utils import MetricsTracker

env = PredatorPreyEnv()
agents = {name: RandomAgent() for name in env.agents}
metrics = MetricsTracker()

for ep in range(500):
    obs = env.reset()
    episode_return = 0
    done = False
    while not done:
        actions = {name: agents[name].select_action(obs[name]) for name in env.agents}
        obs, rewards, dones, infos = env.step(actions)
        episode_return += sum(rewards.values()) / len(env.agents)
        done = dones["__all__"]
    metrics.record(episode_return, infos["step"], infos["captured"], infos["cooperative_capture"])

s = metrics.summary()
print(f"Random baseline (500 episodes):")
print(f"  Capture rate:      {s['capture_rate']:.1%}")
print(f"  Coop capture rate: {s['coop_capture_rate']:.1%}")
print(f"  Mean return:       {s['mean_return']:.2f}")
print(f"  Mean ep length:    {s['mean_length']:.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(metrics.episode_lengths, bins=20, edgecolor="black")
ax1.set_xlabel("Episode Length")
ax1.set_title("Episode Length Distribution")
ax2.bar(["Captured", "Timeout"], [sum(metrics.captures), len(metrics.captures) - sum(metrics.captures)])
ax2.set_title("Outcomes")
plt.tight_layout()

## 4. Load and Compare Trained Agents

After running `train_iql.py` and `train_vdn.py`, load the checkpoints and compare.

In [ ]:
import os

# Load IQL agents (run train_iql.py first)
iql_agents = {}
for name in ["predator_0", "predator_1"]:
    path = f"../checkpoints/iql_{name}.pkl"
    if os.path.exists(path):
        agent = IQLAgent()
        agent.load(path)
        iql_agents[name] = agent
        print(f"Loaded IQL {name}, Q-table size: {len(agent.q_table)}")
    else:
        print(f"Checkpoint not found: {path} (run train_iql.py first)")

# Load VDN agent (run train_vdn.py first)
vdn_path = "../checkpoints/vdn.pt"
if os.path.exists(vdn_path):
    vdn = VDNAgent()
    vdn.load(vdn_path)
    print(f"Loaded VDN agent")
else:
    print(f"Checkpoint not found: {vdn_path} (run train_vdn.py first)")